# 1、ConversationTokenBufferMemory的使用


ConversationTokenBufferMemory 是 LangChain 中一种非常实用且精准的记忆管理策略。

与 ConversationBufferWindowMemory（限制对话轮数）不同，它是根据 Token 的总数量来限制记忆的长度。

核心作用
它通过计算当前记忆中所有消息的 Token 总和，一旦超过设定的 max_token_limit，就会自动删除最早的消息，直到 Token 数量回到安全范围内。

为什么它比 WindowMemory 更好？

防止爆 Context Window：一条包含 1000 个字的消息和一条包含 2 个字的消息，在 WindowMemory 里都算 "1轮"。但在大模型眼中，前者的成本和显存占用是后者的数百倍。
精准控制成本：可以直接根据 LLM 的最大上下文限制（如 4k, 8k, 128k）来配置记忆，最大程度保留最近的对话。

举例1：

In [2]:

import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [9]:
# 1.导入相关包
# ==========================================
# 关键修复点：必须放在其他 LangChain 导入之前
# from langchain_core.caches import BaseCache
# ==========================================
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

# ==========================================
# 关键修复点：手动重建 Pydantic 模型
ConversationTokenBufferMemory.model_rebuild()
# ==========================================

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. 实例化 ConversationTokenBufferMemory
# max_token_limit=50: 设定记忆最多只保留 50 个 Token (为了演示效果设得很小，实际使用通常是 2000+)
# return_messages=True: 返回消息对象列表，适合 ChatModel
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=50,
    return_messages=True
)

# 3. 模拟对话存入
# 第一轮 (Token 很少)
memory.save_context({"input": "你好"}, {"output": "你好呀"})

# 第二轮 (Token 较多，假设这是一段很长的对话)
# 为了演示，我写了一段稍微长一点的文字
long_text = "AI 是人工智能的缩写，它涉及到计算机科学、统计学等多个领域。"
memory.save_context({"input": "什么是AI？"}, {"output": long_text})

# 第三轮 (Token 很少)
memory.save_context({"input": "再见"}, {"output": "拜拜"})

# 4. 查看当前记忆状态
history = memory.load_memory_variables({})
print(f"当前记忆中的消息 (限制 50 token):")
print(history)

PydanticUndefinedAnnotation: name 'Callbacks' is not defined

For further information visit https://errors.pydantic.dev/2.12/u/undefined-annotation


在 LangChain 0.2/0.3+ 的现代架构（LCEL）中，官方推荐的做法是彻底分离“存储”与“逻辑”。

不再使用 ConversationTokenBufferMemory（这是导致 Pydantic 报错的旧组件）。

使用 InMemoryChatMessageHistory 来存储所有原始对话。

使用 trim_messages 来执行 Token 裁剪逻辑（这是新标准）。

这种写法不仅完全规避了 Pydantic/BaseCache 的报错，而且更加清晰、符合新版标准。


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

# 1. 创建大模型实例 (用于计算 Token)
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 定义裁剪器 (Trimmer) - 这就是 "TokenBufferMemory" 的现代替代品
# 它负责计算 token 并决定保留哪些消息
trimmer = trim_messages(
    max_tokens=50,         # 对应旧版的 max_token_limit
    strategy="last",       # 保留最新的消息
    token_counter=llm,     # 使用 LLM 的分词器计算长度
    include_system=True,   # 是否保留系统消息(如果有)
    start_on="human",      # 确保截断后的第一条消息是人类说的(避免对话从一半开始)
)

# 3. 模拟存储 (History) - 替代 memory.save_context
# 在新版中，我们使用 ChatMessageHistory 对象来存取消息
history = InMemoryChatMessageHistory()

# --- 模拟存入对话 ---

# 第一轮 (Token 很少)
history.add_user_message("你好")
history.add_ai_message("你好呀")

# 第二轮 (Token 较多)
long_text = "AI 是人工智能的缩写，它涉及到计算机科学、统计学等多个领域。"
history.add_user_message("什么是AI？")
history.add_ai_message(long_text)

# 第三轮 (Token 很少)
history.add_user_message("再见")
history.add_ai_message("拜拜")

# 4. 查看结果
# 逻辑：先从 history 取出所有消息 -> 扔给 trimmer 进行裁剪 -> 得到最终结果
all_messages = history.messages
trimmed_messages = trimmer.invoke(all_messages)

print(f"原始存储的消息数: {len(all_messages)}")
print("-" * 30)
print(f"Token限制后保留的消息 (限制 50 token):")
print(trimmed_messages)

原始存储的消息数: 6
------------------------------
Token限制后保留的消息 (限制 50 token):
[HumanMessage(content='什么是AI？', additional_kwargs={}, response_metadata={}), AIMessage(content='AI 是人工智能的缩写，它涉及到计算机科学、统计学等多个领域。', additional_kwargs={}, response_metadata={}), HumanMessage(content='再见', additional_kwargs={}, response_metadata={}), AIMessage(content='拜拜', additional_kwargs={}, response_metadata={})]


关键变化解析
没有 Pydantic 报错：
因为 trim_messages 是一个纯函数，InMemoryChatMessageHistory 是基础类，它们不依赖复杂的 Pydantic 继承链。

InMemoryChatMessageHistory: 这是现在的标准存储方式。你可以把它想象成数据库，它负责“存所有数据”。

trim_messages: 这是现在的标准处理方式。它负责“只取最近的 N 个 Token”。

流程解耦：
旧版 Memory 既负责存，又负责删。

新版逻辑中，存储归存储（不丢失数据），裁剪归裁剪（只在发给 LLM 前处理）。

举例2：

In [11]:
# 1.导入相关包
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=20  # 设置token上限，默认值为2000
)

# 添加对话
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
memory.save_context({"input": "今天天气如何？"}, {"output": "晴天，25度"})

# 查看当前记忆
print(memory.load_memory_variables({}))

PydanticUserError: `ConversationTokenBufferMemory` is not fully defined; you should define `BaseCache`, then call `ConversationTokenBufferMemory.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.12/u/class-not-fully-defined



在 LangChain 0.3+ 的最新架构（LCEL）中，官方推荐将 “存储”（History）与 “处理逻辑”（Token 限制）完全解耦。

不再使用 ConversationTokenBufferMemory（它同时负责存储和删除），而是：

使用 InMemoryChatMessageHistory 存储完整的对话。

使用 trim_messages 动态计算并裁剪需要的 Token。

以下是对应的最新写法：


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages

# 1. 创建大模型 (用于计算 Token)
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 定义裁剪器 (Trimmer) - 替代 TokenBufferMemory 的核心逻辑
# 这是一个纯逻辑组件，负责计算 token 并“切掉”旧消息
trimmer = trim_messages(
    max_tokens=20,         # 设置上限 (你的需求)
    strategy="last",       # 保留最新的消息
    token_counter=llm,     # 使用 LLM 的分词器
    include_system=True,   # 建议：如果以后有系统提示词，保留它
    start_on="human",      # 建议：确保切完后的第一句话是人说的，保持对话完整性
)

# 3. 存储实例 - 替代 memory.save_context
# 它像一个数据库，只管存，不负责删
history = InMemoryChatMessageHistory()

# --- 模拟添加对话 ---
# 第一轮
history.add_user_message("你好吗？")
history.add_ai_message("我很好，谢谢！")

# 第二轮
history.add_user_message("今天天气如何？")
history.add_ai_message("晴天，25度")

# --- 查看结果 ---
# 逻辑：从存储中取出所有消息 -> 扔进 trimmer 进行裁剪 -> 得到最终结果
# 注意：20个 token 对于中文来说非常少，可能只能保留最后一句话
trimmed_messages = trimmer.invoke(history.messages)

print(f"原始消息条数: {len(history.messages)}")
print(f"裁剪后结果 (Max 20 Tokens):")
print(trimmed_messages)

原始消息条数: 4
裁剪后结果 (Max 20 Tokens):
[]


为什么现在的写法更好？

非破坏性 (Non-destructive)：
旧写法 (ConversationTokenBufferMemory)：一旦 Token 超限，旧消息就永久被删除了，找不回来了。
新写法：history 里永远存着完整的对话记录（方便做审计、存数据库）。裁剪只发生在 trimmer.invoke() 这一刻（即发给 AI 之前），原数据还在。

没有版本冲突：
完全规避了你之前遇到的 PydanticUserError，因为 trim_messages 是纯函数，不依赖复杂的类继承。

符合 Chat 模型标准：
输出的是 [HumanMessage(...), AIMessage(...)] 对象列表，这是传给 gpt-4o 等 Chat 模型的原生格式，而不是旧版的纯文本字符串。


# 2、ConversationSummaryMemory的使用

ConversationSummaryMemory  是langchain中一种高级的记忆管理方式。

1. 核心概念

与 BufferMemory （原样存储所有对话） 或 WindowMemory （只存最近k轮对话）不同，ConversationSummaryMemory 不会保存原始的对话记录。

相反，它会利用大模型 在后台不断的对话和进行摘要（Summarize）。

- 比喻：它就像一个会议纪要员。它不记录每个人说的每一句话，而是记录：“用户问了关于人工智能的问题，AI 解释了人工智能的定义和应用。”
- 适用场景：非常长的对话，需要记住很久之前的上下文，但不需要记住具体的原话。


2. 代码示例

以下代码展示了它如何工作。请注意，为了运行这段代码，你的环境需要能访问 OpenAI，因为生成摘要本身需要消耗 Token 和调用 API。

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 定义 Chain: Prompt -> LLM -> String
summary_prompt = ChatPromptTemplate.from_template(
    """
    你是一个负责记录对话摘要的助手。
    请根据【新的对话内容】来更新【当前的对话摘要】。
    如果当前摘要为空，请直接根据新对话生成摘要。

    【当前的对话摘要】:
    {summary}

    【新的对话内容】:
    User: {new_input}
    AI: {new_output}

    请生成更新后的摘要 (使用第三人称，保持简洁):
    """
)

summary_chain = summary_prompt | llm | StrOutputParser()

# 初始化摘要状态
current_summary = "（暂无）"

# 辅助函数
def update_memory(previous_summary, user_input, ai_output):
    new_summary = summary_chain.invoke({
        "summary": previous_summary,
        "new_input": user_input,
        "new_output": ai_output
    })
    return new_summary

# --- 测试 ---
print("--- 第 1 轮 ---")
u_in = "你好，我是小明。我是一名软件工程师。"
a_out = "你好小明，很高兴认识你。"

current_summary = update_memory(current_summary, u_in, a_out)
print(f"当前记忆摘要: {current_summary}")

--- 第 1 轮 ---
当前记忆摘要: 用户小明是一名软件工程师，与助手进行了初次交流。


举例2：如果实例化ConversationSummaryMemory前，已经有历史消息，可以调用from_messages()实例化

在现代 LangChain 开发中，如果你有历史消息并想生成摘要，我们不再依赖 ConversationSummaryMemory 这个黑盒类，而是显式地运行一次摘要链。

这种方式逻辑更透明：“加载历史 -> 跑摘要 Chain -> 得到初始 Summary -> 开始新对话”。

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. 准备模型和摘要链
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

summary_prompt = ChatPromptTemplate.from_template(
    """
    请根据以下对话历史生成一段简洁的摘要:
    {history_text}
    """
)
# 定义一个专门生成摘要的工具
initial_summary_chain = summary_prompt | llm | StrOutputParser()

# 2. 准备历史消息
history = InMemoryChatMessageHistory()
history.add_user_message("你好，你是谁？")
history.add_ai_message("我是AI助手小智")

# 3. 模拟 "from_messages" 的行为
# 手动将历史消息转为文本，然后生成初始摘要
def get_history_text(messages):
    return "\n".join([f"{msg.type}: {msg.content}" for msg in messages])

print("--- 正在根据历史消息计算初始摘要... ---")
history_text = get_history_text(history.messages)
current_summary = initial_summary_chain.invoke({"history_text": history_text})

print(f"初始摘要: {current_summary}")

# 4. 继续对话 (LCEL 手动更新逻辑)
# 定义更新摘要的 Chain
update_prompt = ChatPromptTemplate.from_template(
    "当前摘要: {summary}\n新对话:\nUser: {new_input}\nAI: {new_output}\n请更新摘要:"
)
update_chain = update_prompt | llm | StrOutputParser()

# 模拟新一轮对话
user_input = "我的名字叫小明"
ai_output = "很高兴认识你"

# 将新对话加入历史 (可选，看是否需要保留原始记录)
history.add_user_message(user_input)
history.add_ai_message(ai_output)

# 更新摘要
current_summary = update_chain.invoke({
    "summary": current_summary,
    "new_input": user_input,
    "new_output": ai_output
})

print(f"\n更新后的摘要: {current_summary}")

--- 正在根据历史消息计算初始摘要... ---
初始摘要: 人类与AI助手小智进行了简单的问候交流。

更新后的摘要: 摘要: 用户小明与AI助手小智进行了简单的问候交流。



关键点解析

ConversationSummaryMemory.from_messages 的作用：

它不仅是加载了 chat_memory，更重要的是它在实例化的一瞬间，就在后台调用了 LLM，把传入的 history 压缩成了一段 summary。所以初始化可能需要几秒钟（因为要等待 OpenAI 返回）。

chat_memory 参数：
它指定了底层的存储介质。如果不传，它会默认创建一个空的内存列表。传入 history 后，它就拥有了“记忆原材料”。


# 3、ConversationSummaryBufferMemory的使用

ConversationSummaryBufferMemory 是 LangChain 中一种**混合型（Hybrid）**的记忆管理策略。它结合了 ConversationBufferMemory（保留原话）和 ConversationSummaryMemory（保留摘要）的优点。

1. 核心原理
它在内存中维护两个区域：
- Buffer（缓冲区）：保存最近的几轮对话的原话（Raw Text）。这保证了 AI 能精准回答最近的问题（比如“它是什么意思？”中的“它”指代什么）。
- Summary（摘要区）：当 Buffer 中的 Token 数量超过设定的 max_token_limit 时，它不会像 WindowMemory 那样直接丢弃旧消息，而是把旧消息压缩成摘要，存入摘要区。

比喻：就像人类的记忆。你记得刚才那一秒朋友具体说了哪个字（Buffer），但只记得昨天朋友大概说了什么事情（Summary）。

2. 核心优势
- 短期记忆精准：最近的对话保留原汁原味，上下文连贯性极强。
- 长期记忆不丢：久远的对话虽然丢失了细节，但核心信息（如名字、偏好、达成的一致）通过摘要留了下来。
- Token 控制：严格遵守 max_token_limit，防止爆 Token。

举例1：

In [22]:
# # ==========================================
# # 1. 修复 Pydantic 报错 (必须放在最前)
# from langchain_core.caches import BaseCache
# # ==========================================
#
# from langchain.memory import ConversationSummaryBufferMemory
# from langchain_openai import ChatOpenAI
#
# # 2. 创建 LLM
# # 这个 LLM 用于两个地方：
# # A. 在后台生成摘要
# # B. 计算 Token 数量
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
#
# # 3. 实例化 ConversationSummaryBufferMemory
# memory = ConversationSummaryBufferMemory(
#     llm=llm,                # 必需：用于写摘要
#     max_token_limit=50,     # 设定阈值：当 Buffer 内容超过 50 Token 时，触发摘要压缩
#     return_messages=True    # 推荐：返回消息对象列表
# )
#
# # --- 模拟对话 ---
#
# # 1. 存入短对话 (Token 未超限)
# memory.save_context({"input": "你好"}, {"output": "你好，我是助手。"})
#
# print("--- 阶段1: Token未超限，保留原话 ---")
# # 此时 memory 里只有 HumanMessage 和 AIMessage
# print(memory.load_memory_variables({}))
#
#
# # 2. 存入较长对话 (触发限制)
# # 假设这一轮对话加上之前的，超过了 50 Token
# long_input = "请给我详细解释一下量子力学中的薛定谔的猫是此时什么意思？"
# long_output = "薛定谔的猫是一个思想实验，由奥地利物理学家薛定谔提出..."
#
# memory.save_context({"input": long_input}, {"output": long_output})
#
# print("\n--- 阶段2: Token超限，旧消息变摘要 ---")
# # 此时你会发现：
# # 1. 最早的 "你好" 那一轮可能已经消失了，变成了一个 SystemMessage (摘要)。
# # 2. 最近的 "薛定谔的猫" 这一轮保留了原话。
# res = memory.load_memory_variables({})
# print(res)


"""
这是目前 LangChain 0.3+ (LCEL) 架构下，实现 ConversationSummaryBufferMemory（混合记忆：近期保留原话 + 远期生成摘要）的最推荐写法。
这种写法完全抛弃了旧的 Memory 类（彻底解决了 Pydantic 报错），采用了**“显式状态管理”的方式。虽然代码看起来比旧版长一点，但它逻辑透明、完全可控、且不会因为版本更新而崩溃**。
核心思路
我们需要实现一个“修剪函数” (prune_memory)，它的逻辑是：
检查当前历史消息的 Token 数。
如果超过限制：
把最旧的几条消息拿出来。
调用 LLM 把它们压缩进摘要 (Summary)。
从历史记录中删除这几条旧消息。
如果未超过限制：保持原样。
"""


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

# ==========================================
# 1. 初始化配置
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MAX_TOKEN_LIMIT = 50  # 设定阈值：历史记录超过多少 Token 就开始压缩 (演示用 50，实际建议 2000+)

# 模拟数据库存储 (在实际应用中，这里可能是 Redis 或 SQL)
history = InMemoryChatMessageHistory()
current_summary = ""  # 初始摘要为空

# ==========================================
# 2. 定义“摘要生成”工具 (LCEL Chain)
# ==========================================
summary_prompt = ChatPromptTemplate.from_template(
    """
    请逐步更新对话摘要。

    【已有摘要】: {summary}

    【新增的旧对话】(这些对话即将被从短期记忆中移除，请将其归纳到摘要中):
    {new_lines}

    请生成新的摘要:
    """
)
summary_chain = summary_prompt | llm | StrOutputParser()

# ==========================================
# 3. 核心逻辑：内存修剪 (The Pruner)
# ==========================================
def manage_memory(chat_history: InMemoryChatMessageHistory, current_summary: str):
    """
    检查 Token 数量。如果超限，将最早的对话“搬运”到摘要中。
    """
    # 1. 获取当前所有消息
    messages = chat_history.messages
    if not messages:
        return current_summary

    # 2. 计算当前 Token 数 (使用 LLM 自带的方法)
    curr_tokens = llm.get_num_tokens_from_messages(messages)

    # 3. 如果未超限，直接返回，什么都不做
    if curr_tokens <= MAX_TOKEN_LIMIT:
        return current_summary

    # 4. 如果超限，开始循环“移出旧消息并更新摘要”
    # 为了保持对话成对 (User+AI)，我们每次移除 2 条
    while curr_tokens > MAX_TOKEN_LIMIT and len(messages) >= 2:
        # 弹出最早的 User 和 AI 消息
        pruned_msgs = [messages[0], messages[1]] # 获取
        chat_history.messages.pop(0) # 删除 User
        chat_history.messages.pop(0) # 删除 AI

        # 将被删除的消息转为文本
        pruned_text = f"User: {pruned_msgs[0].content}\nAI: {pruned_msgs[1].content}"

        print(f"DEBUG: 触发压缩，归档旧消息 -> {pruned_text.strip()}")

        # 调用 Chain 更新摘要
        # 注意：这里如果 current_summary 为空，传 "无"
        current_summary = summary_chain.invoke({
            "summary": current_summary if current_summary else "无",
            "new_lines": pruned_text
        })

        # 重新获取剩余消息并计算 Token
        messages = chat_history.messages
        curr_tokens = llm.get_num_tokens_from_messages(messages)

    return current_summary

# ==========================================
# 4. 定义主对话 Chain
# ==========================================
# 这里的 Prompt 结构： System(摘要) + History(原话) + Human(新问题)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。以下是关于用户的背景摘要：\n{summary}"),
    MessagesPlaceholder(variable_name="history"), # 这里放剩下的“短期记忆”
    ("human", "{input}"),
])

chat_chain = chat_prompt | llm

# ==========================================
# 5. 模拟运行 (测试效果)
# ==========================================

# 辅助函数：执行一轮对话
def run_conversation(user_input):
    global current_summary

    # A. 生成回复
    # 注意：我们将当前的 history (短期) 和 summary (长期) 一起传给 AI
    response = chat_chain.invoke({
        "summary": current_summary if current_summary else "暂无",
        "history": history.messages,
        "input": user_input
    })

    ai_content = response.content
    print(f"\nAI: {ai_content}")

    # B. 将新的一轮对话加入历史
    history.add_user_message(user_input)
    history.add_ai_message(ai_content)

    # C. 关键步骤：管理内存 (检查是否超限，如果超限则更新 summary 并修剪 history)
    current_summary = manage_memory(history, current_summary)

    # D. 打印状态供观察
    print(f"--- 状态检查 ---")
    print(f"当前摘要(Long Term): {current_summary}")
    print(f"保留的原话(Short Term): {[m.content for m in history.messages]}")
    print("-" * 30)

# --- 开始测试 ---

# 第 1 轮 (Token 很少，保留原话)
print(">>> 第 1 轮")
run_conversation("你好，我是小明。")

# 第 2 轮 (Token 增加，保留原话)
print(">>> 第 2 轮")
run_conversation("我喜欢吃苹果。")

# 第 3 轮 (发送长文本，触发阈值 50 Token)
# 此时，第 1 轮的“你好/我是小明”应该会被挤出去，变成摘要
print(">>> 第 3 轮")
run_conversation("请给我讲一个关于牛顿发现万有引力的长故事，要包含苹果的情节。")

# 第 4 轮 (验证摘要是否生效)
# 此时 AI 应该记得我的名字叫小明 (来自摘要)，也记得刚才讲了牛顿 (来自短期记忆)
print(">>> 第 4 轮")
run_conversation("我刚才说的名字是什么？")

"""

代码逻辑图解
这个“新写法”其实就是手动实现了 SummaryBuffer 的核心思想：
1. 输入：用户发来新消息。
2. 生成：把 Summary (摘要) 和 History (近期记录) 一起扔给 AI 生成回复。
3. 存储：把新消息存入 History。
4. 维护 (Prune)：
    看一眼 History 胖不胖 (Token > 50?)。
    如果胖了，把最老的肉割下来，做成压缩饼干 (Summary)。
    原来的肉扔掉。
为什么推荐这种写法？
1. 没有黑盒：你可以清楚地看到 Token 是怎么算的，消息是怎么被弹出的，摘要是怎么更新的。
2. 兼容性强：完全依赖 langchain-core 的基础组件，不依赖 langchain.memory 里的旧代码，彻底根治 PydanticUserError。
3. 可定制：
    如果你想保留最后 5 轮而不是按 Token 算？修改 manage_memory 里的 while 条件即可。
    如果你想把摘要存到文件里？在 manage_memory 里加一行写文件代码即可。
"""

>>> 第 1 轮

AI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
--- 状态检查 ---
当前摘要(Long Term): 
保留的原话(Short Term): ['你好，我是小明。', '你好，小明！很高兴认识你。有什么我可以帮助你的吗？']
------------------------------
>>> 第 2 轮

AI: 苹果是非常健康的水果，富含维生素和纤维。你喜欢哪种苹果呢？比如说红富士、青苹果还是其他品种？
DEBUG: 触发压缩，归档旧消息 -> User: 你好，我是小明。
AI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
DEBUG: 触发压缩，归档旧消息 -> User: 我喜欢吃苹果。
AI: 苹果是非常健康的水果，富含维生素和纤维。你喜欢哪种苹果呢？比如说红富士、青苹果还是其他品种？
--- 状态检查 ---
当前摘要(Long Term): 【已有摘要】: 用户小明与AI进行了初次交流，AI表示很高兴认识小明并询问是否需要帮助。  
【新摘要】: 用户小明与AI进行了初次交流，AI表示很高兴认识小明并询问是否需要帮助。小明提到他喜欢吃苹果，AI回应说苹果是非常健康的水果，并询问小明喜欢哪种苹果。
保留的原话(Short Term): []
------------------------------
>>> 第 3 轮

AI: 在17世纪的英格兰，牛顿是一位年轻的科学家，正沉浸在对自然界的探索中。那时，牛顿在剑桥大学学习，正值一场瘟疫肆虐，学校关闭，他不得不回到家乡的乡村，开始了他独自的思考和实验。

在一个阳光明媚的下午，牛顿坐在自家的花园里，思考着天体运动和地球上的物体如何相互作用。就在这时，他的目光被一棵苹果树吸引住了。树上挂满了成熟的苹果，微风轻轻摇曳着树枝，几个苹果从树上掉落，砸在了地上。

牛顿看着这些苹果，心中突然闪过一个念头：为什么苹果总是垂直向下掉落，而不是向旁边或向上飞去？这个简单的现象引发了他对重力的深刻思考。他开始思考，是否有一种力量在作用于这些苹果，使它们朝向地面。

牛顿回到书房，开始翻阅他之前的笔记，试图找到解释这一现象的理论。他想到了天体之间的相互作用，尤其是月亮和地球之间的关系。他意识到，月亮在空中绕着地球旋转，而地球上的物体则被一种看不见的力量吸引着。

经过几个月

对比组：


In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ==========================================
# 1. 基础配置
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MAX_TOKEN_LIMIT = 40  # 设定缓冲区阈值 (为了演示设得很小，实际建议 1000+)

# 初始化状态
history = InMemoryChatMessageHistory() # 对应旧版的 chat_memory
current_summary = ""                   # 对应旧版的 moving_summary_buffer

# ==========================================
# 2. 定义“摘要生成器” (替代旧版的后台逻辑)
# ==========================================
summary_prompt = ChatPromptTemplate.from_template(
    """
    请根据已有摘要和即将被移除的旧对话，生成新的摘要。

    【已有摘要】: {summary}
    【被移除的旧对话】: {new_lines}

    请生成一段简洁的合并摘要:
    """
)
summary_chain = summary_prompt | llm | StrOutputParser()

# ==========================================
# 3. 定义“内存管理器” (核心逻辑)
# ==========================================
def manage_memory(history, current_summary):
    """
    检查缓冲区 Token 数。如果超限，弹出最旧的消息并更新摘要。
    """
    # 获取当前消息列表
    messages = history.messages
    if not messages:
        return current_summary

    # 计算 Token (使用模型自带的方法)
    curr_tokens = llm.get_num_tokens_from_messages(messages)

    # 循环裁剪，直到低于阈值 (且保留成对消息)
    while curr_tokens > MAX_TOKEN_LIMIT and len(messages) >= 2:
        # 1. 弹出最早的一轮对话 (User + AI)
        removed_msgs = messages[:2] # 取前两个
        history.messages = messages[2:] # 剩下的放回去

        # 2. 将被删除的消息转为文本
        removed_text = f"User: {removed_msgs[0].content}\nAI: {removed_msgs[1].content}"

        # 3. 调用 Chain 更新摘要
        print(f"DEBUG: 缓冲区已满 ({curr_tokens} > {MAX_TOKEN_LIMIT})，正在压缩: {removed_text.strip()}...")
        current_summary = summary_chain.invoke({
            "summary": current_summary if current_summary else "无",
            "new_lines": removed_text
        })

        # 4. 重新检查 Token
        messages = history.messages
        curr_tokens = llm.get_num_tokens_from_messages(messages)

    return current_summary

# ==========================================
# 4. 模拟运行过程
# ==========================================

# 辅助函数：模拟“存入对话”并自动触发“内存管理”
def save_context_and_process(human_input, ai_output):
    global current_summary

    # A. 存入新对话
    history.add_user_message(human_input)
    history.add_ai_message(ai_output)

    # B. 触发内存管理 (核心！)
    current_summary = manage_memory(history, current_summary)

# --- 开始测试 ---

print(">>> 第1轮: 存入 '你好...'")
save_context_and_process("你好，我的名字叫小明", "很高兴认识你")

print("\n>>> 第2轮: 存入 '李白...'")
save_context_and_process("李白是哪个朝代的诗人", "李白是唐朝的诗人")

print("\n>>> 第3轮: 存入 '苏轼...'")
save_context_and_process("唐宋八大家里有苏轼吗？", "有")

# ==========================================
# 5. 结果对比 (对应旧版的两个 print)
# ==========================================

print("\n" + "="*40)
print("对比结果展示")
print("="*40)

# 场景 A: 构造给 LLM 看的完整 Prompt (对应 old_memory.load_memory_variables)
# 逻辑：SystemMessage(摘要) + List[Message](缓冲区)
full_context_for_llm = [SystemMessage(content=f"对话摘要: {current_summary}")] + history.messages

print("\n[A] 给大模型看的完整上下文 (Summary + Buffer):")
for msg in full_context_for_llm:
    print(f"  - {msg.type}: {msg.content}")

# 场景 B: 查看底层缓冲区 (对应 old_memory.chat_memory.messages)
print("\n[B] 底层缓冲区实际存储 (Buffer Only):")
print(history.messages)

"""
总结
属性  	load_memory_variables({})	            chat_memory.messages
用途	    给 AI 看的	                            内部存储用的
内容构成	摘要 (SystemMessage) + 剩余原话	    仅剩的剩余原话
是否完整	是，逻辑上是完整的历史。	            否，丢失了被压缩的旧消息。
LCEL对应	相当于 Prompt 中的 {history}	        相当于 InMemoryChatMessageHistory

开发建议：
在实际开发中，当你调用 chain.invoke 时，LangChain 内部调用的是 A (load_memory_variables)。你几乎不需要直接操作 B，除非你在做调试或者需要把剩下的消息存入数据库。
"""

>>> 第1轮: 存入 '你好...'

>>> 第2轮: 存入 '李白...'
DEBUG: 缓冲区已满 (48 > 40)，正在压缩: User: 你好，我的名字叫小明
AI: 很高兴认识你...

>>> 第3轮: 存入 '苏轼...'
DEBUG: 缓冲区已满 (48 > 40)，正在压缩: User: 李白是哪个朝代的诗人
AI: 李白是唐朝的诗人...

对比结果展示

[A] 给大模型看的完整上下文 (Summary + Buffer):
  - system: 对话摘要: 用户小明与AI进行了简单的自我介绍，AI表示很高兴认识用户。同时，用户询问李白的朝代，AI回答李白是唐朝的诗人。
  - human: 唐宋八大家里有苏轼吗？
  - ai: 有

[B] 底层缓冲区实际存储 (Buffer Only):
[HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={})]


'\n总结\n属性  \tload_memory_variables({})\t            chat_memory.messages\n用途\t    给 AI 看的\t                            内部存储用的\n内容构成\t摘要 (SystemMessage) + 剩余原话\t    仅剩的剩余原话\n是否完整\t是，逻辑上是完整的历史。\t            否，丢失了被压缩的旧消息。\nLCEL对应\t相当于 Prompt 中的 {history}\t        相当于 InMemoryChatMessageHistory\n\n开发建议：\n在实际开发中，当你调用 chain.invoke 时，LangChain 内部调用的是 A (load_memory_variables)。你几乎不需要直接操作 B，除非你在做调试或者需要把剩下的消息存入数据库。\n'

举例2：模拟客服交互

In [15]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.llm import LLMChain

# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)

# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True
)

# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)

# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")

# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

用户: 你好，我想查询订单12345的状态
客服: 您好！感谢您的咨询。关于订单12345的状态，我会尽快为您查询。请稍等片刻。 

（如果您需要更详细的信息，请提供订单相关的联系方式或其他信息，以便我更好地为您服务。）

用户: 这个订单是上周五下的
客服: 谢谢您提供的信息！我会尽快帮您查询上周五下的订单12345的状态。请稍等片刻。

（如果您有其他问题或需要进一步的帮助，请随时告诉我！）

用户: 我现在急着用，能加急处理吗
客服: 我理解您的着急心情！关于加急处理订单的请求，通常需要联系配送部门进行确认。请您提供一下您的联系方式，我会尽快将您的请求反馈给相关部门，争取为您加急处理。

如果您有其他问题或需要进一步的帮助，请随时告诉我！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题，感谢您更新订单号！我将立即为您查询订单12346的状态。请稍等片刻。

如果您还有其他问题或需要进一步的帮助，请随时告诉我！

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货期限**：一般情况下，您可以在收到商品后的7天内申请退货。
2. **退货条件**：商品必须保持未使用状态，包装完整，附带原始标签和发票。
3. **退货流程**：请您先联系客服申请退货，提供订单号和退货原因，我们会为您生成退货申请并提供相应的退货地址。
4. **退款方式**：退货商品确认无误后，我们会在3-5个工作日内处理退款，退款将按照您原支付方式返还。

如果您有具体的商品需要退货或者其他相关问题，请告诉我，我会尽力为您提供帮助！


=== 当前记忆内容 ===
{'chat_history': [SystemMessage(content='The human inquires about the status of order 12345. The AI responds by thanking the human for their inquiry and states that it will quickly check the status of the order, asking the human to wait a moment. The AI also offers to assist further if the hum

In [25]:
"""
这是一个非常经典的 LangChain 应用场景（客服机器人）。

你原始的代码使用了 `LLMChain` 和 `ConversationSummaryBufferMemory`，这两个在 LangChain 0.3+ 版本中：
1.  `LLMChain` 已经被 **废弃**，官方推荐使用 LCEL (`prompt | llm`)。
2.  `ConversationSummaryBufferMemory` 是旧版组件，容易出现你之前遇到的 `PydanticUserError`，且不够透明。

下面是**基于 LangChain 0.3 (LCEL) 标准**优化后的代码。
它实现了和你原代码完全相同的逻辑：**“混合记忆”**（短期保留原话 + 长期压缩摘要），但更稳定、更现代化，且完全解决了版本兼容性问题。
"""

### 优化后的代码

import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ==========================================
# 1. 初始化配置
# ==========================================
# 初始化大模型 (同时用于对话和生成摘要)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)

# 设定记忆阈值 (为了演示效果，设为 300 token；实际生产环境通常设为 2000+)
MAX_TOKEN_LIMIT = 300

# 初始化状态存储
history = InMemoryChatMessageHistory() # 替代旧版 memory
current_summary = ""                   # 替代旧版 memory.moving_summary_buffer

# ==========================================
# 2. 定义核心组件 (LCEL)
# ==========================================

# A. 定义主对话 Prompt
# 逻辑：系统人设 + 长期摘要 + 短期历史 + 用户输入
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。

    【之前的对话摘要】:
    {summary}
    """),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# B. 定义主对话 Chain
chat_chain = chat_prompt | llm | StrOutputParser()

# C. 定义摘要生成 Chain (后台运行)
summary_prompt = ChatPromptTemplate.from_template(
    """
    请根据已有摘要和即将被移除的旧对话，更新对话摘要。

    【已有摘要】: {summary}
    【即将移除的旧对话】: {new_lines}

    请生成新的摘要:
    """
)
summary_chain = summary_prompt | llm | StrOutputParser()

# ==========================================
# 3. 定义记忆管理逻辑 (The Manager)
# ==========================================
def manage_memory_lifecycle(user_input, ai_output):
    """
    负责：1.存入新消息 2.检查Token超限 3.执行压缩
    """
    global current_summary

    # 1. 存入新对话
    history.add_user_message(user_input)
    history.add_ai_message(ai_output)

    # 2. 检查 Token 是否超限
    curr_tokens = llm.get_num_tokens_from_messages(history.messages)

    # 3. 如果超限，循环修剪旧消息并更新摘要
    # (保留至少 2 条最近的消息作为 Buffer)
    while curr_tokens > MAX_TOKEN_LIMIT and len(history.messages) > 2:
        # 弹出最早的一组对话 (User + AI)
        pruned_msgs = history.messages[:2]
        history.messages = history.messages[2:]

        pruned_text = f"User: {pruned_msgs[0].content}\nAI: {pruned_msgs[1].content}"

        # 更新摘要
        print(f"   [系统日志] 记忆已满 ({curr_tokens} tokens)，正在将旧对话压缩进摘要...")
        current_summary = summary_chain.invoke({
            "summary": current_summary if current_summary else "无",
            "new_lines": pruned_text
        })

        # 重新计算 Token
        curr_tokens = llm.get_num_tokens_from_messages(history.messages)

# ==========================================
# 4. 模拟多轮对话
# ==========================================
dialogue = [
    "你好，我想查询订单12345的状态",
    "这个订单是上周五下的",
    "我现在急着用，能加急处理吗",
    "等等，我可能记错订单号了，应该是12346",
    "对了，你们退货政策是怎样的",
    "谢谢，那12346这个订单帮我备注一下加急"
]

print("=== 客服对话开始 ===\n")

for i, user_input in enumerate(dialogue):
    print(f"--- 第 {i+1} 轮 ---")

    # 1. 执行对话 (传入 Summary + History)
    response_text = chat_chain.invoke({
        "summary": current_summary if current_summary else "（暂无历史）",
        "history": history.messages,
        "input": user_input
    })

    print(f"用户: {user_input}")
    print(f"客服: {response_text}")

    # 2. 触发记忆管理 (存入+压缩)
    manage_memory_lifecycle(user_input, response_text)
    print("")

# ==========================================
# 5. 查看最终记忆状态
# ==========================================
print("=== 最终记忆状态检查 ===")
print(f"1. 长期记忆 (Summary):\n   {current_summary}")
print(f"\n2. 短期记忆 (Buffer - 仅保留原话):\n   {[msg.content for msg in history.messages]}")


"""
### 优化点解析

1.  **移除 `LLMChain`**:
    *   旧代码：`chain = LLMChain(...)`
    *   新代码：`chat_chain = prompt | llm | StrOutputParser()`
    *   **优势**：符合 LCEL 标准，管道式调用，结构更清晰。

2.  **解耦 `Memory`**:
    *   旧代码：使用黑盒的 `ConversationSummaryBufferMemory`，容易报 Pydantic 错，且内部 Token 计算逻辑不可见。
    *   新代码：
        *   使用 `InMemoryChatMessageHistory` 存短期。
        *   使用 `current_summary` 变量存长期。
        *   使用 `manage_memory_lifecycle` 函数显式控制何时压缩。
    *   **优势**：**完全透明**。你可以清楚地看到 Token 何时超限，摘要何时更新，不会因为 LangChain 升级而导致代码崩溃。

3.  **Prompt 结构优化**:
    *   旧代码：Prompt 只有 `{chat_history}`，摘要和原话混在一起。
    *   新代码：Prompt 明确区分了 `System Message` 里的 `Recall/Summary` 和 `MessagesPlaceholder` 里的 `Short-term Context`。这对于 GPT 模型来说，指令依从性更好。

4.  **模拟效果**:
    *   我在循环中加入了 `print` 日志。你会看到，当前几轮对话时，摘要为空；当对话进行到一半（超过 300 Token）时，系统会自动提示“正在压缩旧对话”，将订单号等信息转入 Summary。

### 运行预期
你会发现，即使对话变得很长，短期记忆（Buffer）里只保留最近两三句，但客服依然知道之前提到的“订单号 12346”和“加急需求”，因为这些信息被自动压缩进了 Summary。
"""



=== 客服对话开始 ===

--- 第 1 轮 ---
用户: 你好，我想查询订单12345的状态
客服: 您好！感谢您联系我们。关于订单12345的状态，我将为您查询一下。请稍等片刻。 

（这里可以根据实际情况查询订单状态并回复用户。）

--- 第 2 轮 ---
用户: 这个订单是上周五下的
客服: 谢谢您提供的信息！我会尽快为您查询上周五下的订单12345的状态，请稍等片刻。 

（此处可以根据实际情况查询并回复用户订单状态。）

--- 第 3 轮 ---
用户: 我现在急着用，能加急处理吗
客服: 非常理解您的急需，我们会尽量帮助您加急处理订单。请您提供一下您的联系方式，我们会尽快与相关部门沟通，争取加快发货速度。谢谢您的理解与耐心！

--- 第 4 轮 ---
用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题！感谢您更正订单号。让我为您查询订单12346的状态。请稍等片刻。 

（根据实际情况查询并回复用户订单状态。）

--- 第 5 轮 ---
用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货期限**：一般情况下，您可以在收到商品后7天内申请退货。
2. **退货条件**：商品需保持原包装、未使用且附带所有标签和配件。
3. **申请流程**：请您登录账户，找到订单，选择需要退货的商品并提交退货申请，我们会尽快审核。
4. **退款方式**：退款将原路退回到您的支付账户，处理时间一般为3-5个工作日。

如果您还有其他问题或者需要具体的帮助，请随时告诉我！
   [系统日志] 记忆已满 (393 tokens)，正在将旧对话压缩进摘要...
   [系统日志] 记忆已满 (337 tokens)，正在将旧对话压缩进摘要...

--- 第 6 轮 ---
用户: 谢谢，那12346这个订单帮我备注一下加急
客服: 好的，我会为您的订单12346备注加急处理。请您放心，我们会尽力加快处理速度。感谢您的耐心与理解，如果还有其他需要帮助的地方，请随时告诉我！
   [系统日志] 记忆已满 (342 tokens)，正在将旧对话压缩进摘要...

=== 最终记忆状态检查 ===
1. 长期记忆 (Summary):
   【新摘要】: 用户询问订单12345的状态，并表示急需用货，希望加急处理。

'\n### 优化点解析\n\n1.  **移除 `LLMChain`**:\n    *   旧代码：`chain = LLMChain(...)`\n    *   新代码：`chat_chain = prompt | llm | StrOutputParser()`\n    *   **优势**：符合 LCEL 标准，管道式调用，结构更清晰。\n\n2.  **解耦 `Memory`**:\n    *   旧代码：使用黑盒的 `ConversationSummaryBufferMemory`，容易报 Pydantic 错，且内部 Token 计算逻辑不可见。\n    *   新代码：\n        *   使用 `InMemoryChatMessageHistory` 存短期。\n        *   使用 `current_summary` 变量存长期。\n        *   使用 `manage_memory_lifecycle` 函数显式控制何时压缩。\n    *   **优势**：**完全透明**。你可以清楚地看到 Token 何时超限，摘要何时更新，不会因为 LangChain 升级而导致代码崩溃。\n\n3.  **Prompt 结构优化**:\n    *   旧代码：Prompt 只有 `{chat_history}`，摘要和原话混在一起。\n    *   新代码：Prompt 明确区分了 `System Message` 里的 `Recall/Summary` 和 `MessagesPlaceholder` 里的 `Short-term Context`。这对于 GPT 模型来说，指令依从性更好。\n\n4.  **模拟效果**:\n    *   我在循环中加入了 `print` 日志。你会看到，当前几轮对话时，摘要为空；当对话进行到一半（超过 300 Token）时，系统会自动提示“正在压缩旧对话”，将订单号等信息转入 Summary。\n\n### 运行预期\n你会发现，即使对话变得很长，短期记忆（Buffer）里只保留最近两三句，但客服依然知道之前提到的“订单号 12346”和“加急需求”，因为这些信息被自动压缩进了 Summary。\n'

# 4、ConversationEntityMemory的使用（了解）


ConversationEntityMemory 介绍与使用

ConversationEntityMemory 是 LangChain 中一种结构化的记忆类型。

1. 核心概念

与其他简单记录“对话流水账”的 Memory 不同，ConversationEntityMemory 的目标是提取实体（Entity）并建立知识库。

2. 它的工作方式：
- 提取：当用户说话时，它会在后台调用 LLM，自动识别话语中的“实体”（如人名、地名、公司名、特定名词）。
- 存储：它不仅仅存对话历史，还在内存中维护一个 HashMap (字典)，记录每个实体的相关事实（Facts）。
- 注入：当下一轮对话提到某个实体时，它会将该实体的所有已知事实注入到 Prompt 中，作为上下文。


3. 适用场景：
- 需要记住关于特定人物、物品详细信息的对话。
- 构建类似 RPG 游戏中的 NPC 记忆（记住玩家的属性、任务状态）。
- 构建复杂的个人助理（记住用户的喜好、生日、职业）。


In [37]:
from langchain.chains.conversation.base import LLMChain
from langchain.memory import ConversationEntityMemory
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from langchain_openai import ChatOpenAI

# 初始化大语言模型
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)
# 使用LangChain为实体记忆设计的预定义模板
prompt = ENTITY_MEMORY_CONVERSATION_TEMPLATE
# 初始化实体记忆
memory = ConversationEntityMemory(llm=llm)
# 提供对话链
chain = LLMChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=ConversationEntityMemory(llm=llm),
    #verbose=True,  # 设置为True可以看到链的详细推理过程
)

# 进行几轮对话，记忆组件会在后台自动提取和存储实体信息
chain.invoke(input="你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。")
chain.invoke(input="我住在纽约。")
chain.invoke(input="我使用的装备是由斯塔克工业提供的。")

# 查询记忆体中存储的实体信息
print("\n当前存储的实体信息:")
print(chain.memory.entity_store.store)




当前存储的实体信息:
{'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。', '钢铁侠': '钢铁侠是蜘蛛侠的好朋友之一。', '美国队长': '美国队长是蜘蛛侠的好朋友之一。', '绿巨人': '绿巨人是蜘蛛侠的好朋友之一。', '纽约': '蜘蛛侠住在纽约。', '斯塔克工业': '斯塔克工业提供了蜘蛛侠使用的装备。'}


In [38]:
# 基于记忆进行提问
answer = chain.invoke(input="你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？")
print("\nAI的回答:")
print(answer)


AI的回答:
{'input': '你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？', 'history': 'Human: 你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。\nAI: 你好，蜘蛛侠！很高兴认识你。你和钢铁侠、美国队长以及绿巨人都是超级英雄，真是一个强大的团队！你们最近有什么冒险吗？\nHuman: 我住在纽约。\nAI: 纽约是一个充满活力的城市，适合超级英雄们活动！你在纽约的生活怎么样？有没有遇到什么有趣的事情或者挑战？\nHuman: 我使用的装备是由斯塔克工业提供的。\nAI: 斯塔克工业的装备真是太棒了！钢铁侠的技术总是让人惊叹。你最喜欢使用哪一件装备？它在你的冒险中帮助了你哪些方面？', 'entities': {'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。', '纽约': '蜘蛛侠住在纽约。', '钢铁侠': '钢铁侠是蜘蛛侠的好朋友之一。', '美国队长': '美国队长是蜘蛛侠的好朋友之一。', '绿巨人': '绿巨人是蜘蛛侠的好朋友之一。'}, 'text': '蜘蛛侠住在纽约。他的好朋友包括钢铁侠、美国队长和绿巨人。这些超级英雄们常常一起合作，面对各种挑战和敌人。你对他们的冒险有什么特别的记忆吗？'}


In [ ]:
"""
现代替代方案 (LangGraph / LCEL)
在最新的开发实践中，如果你需要实现类似的功能，通常不再使用 ConversationEntityMemory，而是使用 Tool Calling (工具调用) / Structured Output。
思路如下：
定义一个 Pydantic 数据模型（例如 UserProfile，包含 name, job, likes）。
使用 llm.with_structured_output(UserProfile)。
每当用户说话，让 LLM 提取这些字段，并存入数据库或 JSON 文件。
在 Prompt 中只加载相关的 JSON 数据。
这样做比 ConversationEntityMemory 更精准、更可控，且不会出现奇怪的提取错误。
"""

"""
这是目前最前沿、最符合 LangChain 0.3+ 标准的实现方式。
我们不再使用“黑盒”的 ConversationEntityMemory，而是使用 LangGraph 来构建一个工作流。在这个工作流中，我们明确定义两个步骤：
信息提取 (Extractor)：使用 LLM 的 结构化输出 (Structured Output) 功能，精准提取实体和事实。
回复生成 (Responder)：将提取到的“知识库”注入到 Prompt 中，生成回复。
这种方法的优势是：精准、可控、类型安全（Pydantic）、且完全透明。
"""




## 完整代码实现 (LangGraph + Pydantic v2)
请确保安装了 langgraph：


In [3]:
from typing import List, Dict, Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END

# ==========================================
# 1. 定义数据结构 (Schema)
# ==========================================

# 定义我们要提取的“实体”结构
class Entity(BaseModel):
    name: str = Field(description="实体名称，如人名、公司名、特定名词")
    fact: str = Field(description="关于该实体的核心事实或属性")

# 定义提取结果的列表
class EntityExtraction(BaseModel):
    entities: List[Entity]

# 定义 Graph 的状态 (State)
# 这相当于在整个工作流中流转的“共享内存”
class AgentState(TypedDict):
    messages: List[BaseMessage]     # 对话历史
    knowledge_base: Dict[str, str]  # 实体知识库 {name: fact}
    current_input: str              # 当前用户输入

# ==========================================
# 2. 初始化组件
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- A. 实体提取器 (Extractor) ---
# 使用 with_structured_output 强制模型输出 JSON 格式
extractor_llm = llm.with_structured_output(EntityExtraction)

extract_prompt = ChatPromptTemplate.from_template(
    """
    你是一个信息提取专家。
    请从用户的输入中分析并提取关键实体及其相关事实。

    【已有知识库】(用于参考，避免重复或矛盾):
    {existing_knowledge}

    【用户输入】:
    {input}

    如果用户输入中包含新的事实，请提取出来。如果没有新实体，返回空列表。
    """
)
extractor_chain = extract_prompt | extractor_llm

# --- B. 对话回复器 (Responder) ---
respond_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是一个智能助手。请基于【知识库】和【对话历史】回答用户。

    【当前知识库】:
    {knowledge_context}
    """),
    ("placeholder", "{messages}"), # 自动填充历史消息
    ("human", "{input}")
])
respond_chain = respond_prompt | llm | StrOutputParser()

# ==========================================
# 3. 定义节点逻辑 (Nodes)
# ==========================================

def extract_node(state: AgentState):
    """
    节点1：分析输入，提取实体，更新知识库
    """
    current_kb = state.get("knowledge_base", {})
    user_input = state["current_input"]

    # 将现有知识库转为字符串，供提取器参考
    kb_str = "\n".join([f"{k}: {v}" for k, v in current_kb.items()])

    # 调用提取链
    extraction_result = extractor_chain.invoke({
        "existing_knowledge": kb_str,
        "input": user_input
    })

    # 更新知识库 (Merge Logic)
    # 这里做一个简单的覆盖/追加逻辑
    if extraction_result and extraction_result.entities:
        print(f"   [提取器] 发现新实体: {extraction_result.entities}")
        for entity in extraction_result.entities:
            # 如果实体已存在，简单的拼接新事实 (实际业务中可以用 LLM 做更智能的合并)
            if entity.name in current_kb:
                current_kb[entity.name] += f"; {entity.fact}"
            else:
                current_kb[entity.name] = entity.fact
    else:
        print("   [提取器] 未发现新实体")

    # 返回更新后的状态
    return {"knowledge_base": current_kb}

def respond_node(state: AgentState):
    """
    节点2：生成回复
    """
    current_kb = state.get("knowledge_base", {})
    messages = state.get("messages", [])
    user_input = state["current_input"]

    # 格式化知识库给 Prompt
    kb_context = "\n".join([f"- {k}: {v}" for k, v in current_kb.items()])
    if not kb_context:
        kb_context = "(暂无已知信息)"

    # 生成回复
    response_text = respond_chain.invoke({
        "knowledge_context": kb_context,
        "messages": messages,
        "input": user_input
    })

    # 更新对话历史 (把这一轮 User 和 AI 的话加进去)
    new_messages = messages + [
        HumanMessage(content=user_input),
        AIMessage(content=response_text)
    ]

    return {"messages": new_messages}

# ==========================================
# 4. 构建图 (Graph Construction)
# ==========================================
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("extractor", extract_node)
workflow.add_node("responder", respond_node)

# 设置边 (流程方向)
# Start -> Extractor -> Responder -> End
workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "responder")
workflow.add_edge("responder", END)

# 编译图
app = workflow.compile()

# ==========================================
# 5. 模拟运行
# ==========================================

# 初始化全局状态
# 注意：在 LangGraph 中，通常状态是流转的。为了模拟多轮对话，我们手动维护一个 config 或外部变量
state_snapshot = {
    "messages": [],
    "knowledge_base": {},
    "current_input": ""
}

def run_chat(user_text):
    global state_snapshot
    print(f"\n>>> 用户: {user_text}")

    # 更新当前输入
    state_snapshot["current_input"] = user_text

    # 运行图
    # app.invoke 会跑完整个流程并返回最终状态
    final_state = app.invoke(state_snapshot)

    # 获取 AI 回复 (最后一条消息)
    ai_response = final_state["messages"][-1].content
    print(f"AI: {ai_response}")

    # 更新外部状态快照，以便下一轮使用
    state_snapshot = final_state

    # 打印当前的知识库 (Debug)
    print(f"--- 当前知识库: {final_state['knowledge_base']}")

# --- 测试对话 ---

# 1. 提供信息
run_chat("小明是 Google 的高级工程师，他喜欢写 Python 代码。")

# 2. 只有问题 (测试 AI 是否利用了知识库)
run_chat("小明最擅长什么编程语言？")

# 3. 补充新信息
run_chat("他的朋友小红在 SpaceX 工作。")

# 4. 综合查询
run_chat("小明和小红分别在哪家公司？")


>>> 用户: 小明是 Google 的高级工程师，他喜欢写 Python 代码。
   [提取器] 发现新实体: [Entity(name='小明', fact='是 Google 的高级工程师'), Entity(name='Google', fact='是一家科技公司'), Entity(name='Python', fact='是一种编程语言')]
AI: 小明作为 Google 的高级工程师，喜欢写 Python 代码，这表明他在编程方面有很强的技能。Python 是一种广泛使用的编程语言，适合用于各种应用，包括数据分析、人工智能和网络开发等。小明的工作可能涉及到这些领域的项目。
--- 当前知识库: {'小明': '是 Google 的高级工程师', 'Google': '是一家科技公司', 'Python': '是一种编程语言'}

>>> 用户: 小明最擅长什么编程语言？
   [提取器] 未发现新实体
AI: 根据当前知识库，小明是 Google 的高级工程师，并且他喜欢写 Python 代码。因此，可以推测小明最擅长的编程语言是 Python。
--- 当前知识库: {'小明': '是 Google 的高级工程师', 'Google': '是一家科技公司', 'Python': '是一种编程语言'}

>>> 用户: 他的朋友小红在 SpaceX 工作。
   [提取器] 发现新实体: [Entity(name='小红', fact='在 SpaceX 工作'), Entity(name='SpaceX', fact='是一家科技公司')]
AI: 是的，小红在 SpaceX 工作。SpaceX 是一家科技公司，专注于航天技术和太空探索。小明和小红在各自的公司中都从事与科技相关的工作。
--- 当前知识库: {'小明': '是 Google 的高级工程师', 'Google': '是一家科技公司', 'Python': '是一种编程语言', '小红': '在 SpaceX 工作', 'SpaceX': '是一家科技公司'}

>>> 用户: 小明和小红分别在哪家公司？
   [提取器] 发现新实体: [Entity(name='小明', fact='在 Google 工作'), Entity(name='小红', fact='在 SpaceX 


## 代码核心解析
1. Pydantic 定义 (class Entity):
我们明确告诉 LLM 我们需要 name 和 fact。这比旧版 ConversationEntityMemory 模糊的 Prompt 提取要精准得多。
extractor_llm = llm.with_structured_output(EntityExtraction) 是 LangChain 0.3 处理结构化数据的标准方式。
2. Graph 状态 (AgentState):
我们将“对话历史”(messages)和“知识库”(knowledge_base) 分开存储。
knowledge_base 是一个字典，这和旧版的 entity_store 类似，但现在你是完全可见、可操作的。
3. 双节点架构:
Extractor Node: 专门负责“读”。它只看输入，不生成回复，只更新 knowledge_base。
Responder Node: 专门负责“写”。它读取 knowledge_base 作为背景上下文，生成流畅的回复。

# 5、ConversationKGMemory的使用（了解）


ConversationKGMemory 介绍与使用

ConversationKGMemory (Knowledge Graph Memory) 是 LangChain 中一种基于 知识图谱 (Knowledge Graph) 的记忆机制。
1. 核心概念
与 EntityMemory（记录实体的属性）不同，KGMemory 专注于记录 实体之间的关系 (Relationships)。它将对话转化为 “三元组” (Triplets) 结构：
(主体 Subject, 谓语 Predicate, 客体 Object)
2. 例子：
用户说："小明喜欢吃苹果。"
KG 提取：(小明, 喜欢, 苹果)
用户说："苹果产自山东。"
KG 提取：(苹果, 产自, 山东)
推导：当用户问 "小明喜欢的食物产自哪里？" 时，KG 可以通过链路找到 山东。


In [4]:
#1.导入相关包
from langchain.memory import ConversationKGMemory
from langchain.chat_models import ChatOpenAI

# 2.定义LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3.定义ConversationKGMemory对象
memory = ConversationKGMemory(llm=llm)

# 4.保存会话
memory.save_context({"input": "向山姆问好"}, {"output": "山姆是谁"})
memory.save_context({"input": "山姆是我的朋友"}, {"output": "好的"})

# 5.查询会话
memory.load_memory_variables({"input": "山姆是谁"})

/var/folders/gd/xcfqj9752391g13gs68sxcfr0000gn/T/ipykernel_27738/2292086082.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


PydanticUserError: `ConversationKGMemory` is not fully defined; you should define `BaseCache`, then call `ConversationKGMemory.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.12/u/class-not-fully-defined

In [41]:
memory.get_knowledge_triplets("她最喜欢的颜色是红色")

[KnowledgeTriple(subject='山姆', predicate='是', object_='我的朋友'),
 KnowledgeTriple(subject='山姆', predicate='最喜欢的颜色是', object_='红色')]

In [5]:
"""
现代推荐写法 (LangGraph + 结构化输出)
"""

from typing import List, Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import networkx as nx # 前置依赖：你需要安装 networkx 图算法库： pip install networkx
import matplotlib.pyplot as plt # 可选：用于画图

# ==========================================
# 1. 定义数据结构 (Schema)
# ==========================================
class Triplet(BaseModel):
    subject: str = Field(description="主体，如 Person, Place")
    predicate: str = Field(description="关系，如 loves, works_at, located_in")
    object: str = Field(description="客体，如 Apple, Google, Beijing")

class KnowledgeExtraction(BaseModel):
    triplets: List[Triplet]

# ==========================================
# 2. 初始化组件
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# A. 提取器：强制 LLM 输出三元组
extractor_llm = llm.with_structured_output(KnowledgeExtraction)
extract_prompt = ChatPromptTemplate.from_template(
    """
    你是一个知识图谱构建专家。
    请将用户的输入转化为知识三元组 (Subject, Predicate, Object)。

    用户输入: {input}
    """
)
extractor_chain = extract_prompt | extractor_llm

# B. 全局图谱对象 (模拟数据库)
# 使用 NetworkX 在内存中存储图
G = nx.DiGraph()

# ==========================================
# 3. 核心逻辑函数
# ==========================================

def update_graph(user_input: str):
    """提取三元组并更新图谱"""
    result = extractor_chain.invoke({"input": user_input})

    if result and result.triplets:
        print(f"   [提取中] 发现关系: {result.triplets}")
        for t in result.triplets:
            # 在 NetworkX 图中添加边：Subject -> Object (Label=Predicate)
            G.add_edge(t.subject, t.object, relation=t.predicate)
    else:
        print("   [提取中] 未发现明确关系")

def query_graph(entity: str):
    """简单的图检索：查找与该实体直接相连的邻居"""
    context = []
    if entity in G:
        # 查找出度 (主动关系)
        for neighbor in G.successors(entity):
            relation = G[entity][neighbor]['relation']
            context.append(f"{entity} {relation} {neighbor}")
        # 查找入度 (被动关系)
        for predecessor in G.predecessors(entity):
            relation = G[predecessor][entity]['relation']
            context.append(f"{predecessor} {relation} {entity}")

    return "\n".join(context) if context else "暂无相关图谱信息"

# ==========================================
# 4. 模拟运行
# ==========================================

print(">>> 1. 存入信息: 'Elon Musk created SpaceX'")
update_graph("Elon Musk created SpaceX.")

print("\n>>> 2. 存入信息: 'SpaceX is located in Texas'")
update_graph("SpaceX is located in Texas.")

# 此时图谱逻辑链：Elon Musk -> created -> SpaceX -> located_in -> Texas

print("\n>>> 3. 查询回答")
question = "Elon Musk 的公司位于哪里？"

# 第一步：识别问题中的关键实体 (简单起见，我们假设识别到了 SpaceX)
# 实际项目中可以用 LLM 提取问题中的实体
target_entity = "SpaceX"

# 第二步：检索图谱
kg_context = query_graph(target_entity)
print(f"[检索到的图谱上下文]:\n{kg_context}")

# 第三步：生成回答
answer_prompt = ChatPromptTemplate.from_template(
    "基于知识图谱信息回答问题。\nKG信息: {kg}\n问题: {q}"
)
answer_chain = answer_prompt | llm | StrOutputParser()
res = answer_chain.invoke({"kg": kg_context, "q": question})

print(f"\nAI 回答: {res}")

>>> 1. 存入信息: 'Elon Musk created SpaceX'
   [提取中] 发现关系: [Triplet(subject='Elon Musk', predicate='created', object='SpaceX')]

>>> 2. 存入信息: 'SpaceX is located in Texas'
   [提取中] 发现关系: [Triplet(subject='SpaceX', predicate='located_in', object='Texas')]

>>> 3. 查询回答
[检索到的图谱上下文]:
SpaceX located_in Texas
Elon Musk created SpaceX

AI 回答: Elon Musk 的公司 SpaceX 位于德克萨斯州（Texas）。


## 总结
- ConversationKGMemory: 是旧时代的产物，基于 NetworkX 的简易封装，适合学习原理，不适合生产。
- 现代推荐 (GraphRAG): 使用 llm.with_structured_output 提取三元组，存入图数据库（如 Neo4j, NebulaGraph）或内存图（NetworkX），然后在回答时检索相关联的子图作为 Context。这能提供极其强大的逻辑推理能力。

# 6、VectorStoreRetrieverMemory的使用（了解）

VectorStoreRetrieverMemory 是 LangChain 中一种基于向量检索的记忆机制。

1. 核心概念：无限记忆
之前的 BufferMemory（记流水账）或 SummaryMemory（写摘要）都有一个共同痛点：Token 限制。你无法把一本书的内容塞进 Prompt 的历史记录里。
VectorStoreRetrieverMemory 的解决思路是：并不把所有历史都给大模型看，而是把所有对话历史**向量化（Embedding）**存入向量数据库（Vector Store）。

当用户问新问题时：
系统将用户的问题转为向量。
在数据库中搜索语义最相似的K条历史记录。
只把这K条相关的历史作为 Context 传给大模型。
2. 比喻：它不像是一个记性很好的朋友（记得所有事），而更像是一个手持“搜索引擎”的助理。你问它“去年的财报”，它就去搜“财报”相关的文件给你，而不会把去年所有的会议记录都背一遍。


In [6]:
# 1.导入相关包
from langchain_openai import OpenAIEmbeddings
from langchain.memory import VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory

# 2.定义ConversationBufferMemory对象
memory = ConversationBufferMemory()
memory.save_context({"input": "我最喜欢的食物是披萨"}, {"output": "很高兴知道"})
memory.save_context({"Human": "我喜欢的运动是跑步"}, {"AI": "好的,我知道了"})
memory.save_context({"Human": "我最喜欢的运动是足球"}, {"AI": "好的,我知道了"})

# 3.定义向量嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

# 4.初始化向量数据库
vectorstore = FAISS.from_texts(memory.buffer.split("\n"), embeddings_model)  # 空初始化

# 5.定义检索对象
retriever = vectorstore.as_retriever(search_kwargs=dict(k=1))

# 6.初始化VectorStoreRetrieverMemory
memory = VectorStoreRetrieverMemory(retriever=retriever)

print(memory.load_memory_variables({"prompt": "我最喜欢的食物是"}))

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

这是一个使用 Milvus 作为后端存储来实现 无限向量记忆 (Vector Memory) 的完整代码示例。

我们采用 LangChain 0.3+ (LCEL) 的现代写法：

存储：将对话历史向量化后写入 Milvus。
检索：根据用户问题，从 Milvus 中检索最相关的历史片段 (RAG)。

前置准备
你需要安装 Milvus 的 Python 客户端和 LangChain 适配器：

pip install pymilvus langchain-milvus langchain-openai

In [7]:
import time
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_milvus import Milvus
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. 配置信息
# ==========================================
MILVUS_HOST = "192.168.0.173"
MILVUS_PORT = "19530"
COLLECTION_NAME = "chat_memory_demo"  # Milvus 中的集合名称

# 初始化 Embedding 模型 (用于向量化)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 初始化 LLM (用于对话)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==========================================
# 2. 连接 Milvus 向量数据库
# ==========================================
print(f"正在连接 Milvus ({MILVUS_HOST}:{MILVUS_PORT})...")

# 实例化 Milvus VectorStore
vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={
        "host": MILVUS_HOST,
        "port": MILVUS_PORT
    },
    collection_name=COLLECTION_NAME,
    auto_id=True,        # 自动生成主键 ID
    drop_old=True        # 【注意】演示时设为True会清空旧数据；生产环境请设为False
)

# 创建检索器 (Top-K = 2，只找回最相关的2条历史)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print(f"Milvus 连接成功，集合: {COLLECTION_NAME}")

# ==========================================
# 3. 定义 RAG 对话链 (Retrieval-Augmented Generation)
# ==========================================
prompt = ChatPromptTemplate.from_template(
    """
    你是一个智能助手。请基于以下【相关的历史记忆】来回答用户的问题。
    如果历史记忆中没有相关信息，请忽略它，直接回答问题。

    【相关的历史记忆】:
    {context}

    【用户当前问题】:
    {input}
    """
)

# LCEL 链
chain = prompt | llm | StrOutputParser()

# ==========================================
# 4. 定义记忆管理函数 (存 + 取)
# ==========================================

def save_to_milvus(user_input, ai_output):
    """
    将一轮对话合并为一个 Document，向量化后存入 Milvus
    """
    # 格式化存储内容
    text_content = f"User: {user_input}\nAI: {ai_output}"

    # 封装为 Document 对象
    # metadata 可以存时间戳，方便以后做基于时间的过滤
    doc = Document(
        page_content=text_content,
        metadata={"timestamp": time.time(), "type": "chat_history"}
    )

    # 写入 Milvus
    vector_store.add_documents([doc])
    print(f"   [已写入 Milvus]: {text_content[:20]}...")

def chat_with_milvus_memory(user_input):
    """
    1. 检索 Milvus
    2. 生成回答
    3. 存入新的一轮
    """
    print(f"\n>>> 用户: {user_input}")

    # --- A. 检索 (Retrieve) ---
    # 去 Milvus 查有没有跟这句话相似的历史
    docs = retriever.invoke(user_input)

    # 处理检索结果
    if docs:
        context_text = "\n---\n".join([d.page_content for d in docs])
        print(f"   [Milvus 命中历史]: 找到了 {len(docs)} 条相关记录")
    else:
        context_text = "（暂无相关历史）"
        print("   [Milvus]: 没有找到相关历史")

    # --- B. 生成 (Generate) ---
    response = chain.invoke({
        "context": context_text,
        "input": user_input
    })

    print(f"AI: {response}")

    # --- C. 存储 (Save) ---
    save_to_milvus(user_input, response)

# ==========================================
# 5. 模拟运行
# ==========================================

# 场景 1: 第一次交互 (存入关于爱好的信息)
chat_with_milvus_memory("你好，我叫大卫，我是一名喜欢滑雪的建筑师。")

# 场景 2: 聊点别的 (存入关于饮食的信息)
chat_with_milvus_memory("今天我想吃火锅。")

# 场景 3: 验证记忆 (检索)
# Milvus 会计算 "职业" 和 "爱好" 的向量相似度，自动把第一轮对话找出来
chat_with_milvus_memory("还记得我是做什么工作的吗？")

# 场景 4: 再次验证 (检索)
# Milvus 会把 "吃火锅" 那条找出来
chat_with_milvus_memory("我刚才说想吃什么？")

正在连接 Milvus (192.168.0.173:19530)...
Milvus 连接成功，集合: chat_memory_demo

>>> 用户: 你好，我叫大卫，我是一名喜欢滑雪的建筑师。
   [Milvus]: 没有找到相关历史
AI: 你好，大卫！很高兴认识你。滑雪和建筑都是很有趣的爱好和职业。你最喜欢的滑雪地点是哪里？或者你在建筑方面有什么特别的项目吗？
   [已写入 Milvus]: User: 你好，我叫大卫，我是一名喜欢...

>>> 用户: 今天我想吃火锅。
   [Milvus 命中历史]: 找到了 1 条相关记录
AI: 火锅是个不错的选择！你喜欢什么样的火锅？是麻辣火锅还是清汤火锅？还有你喜欢的配菜有哪些？
   [已写入 Milvus]: User: 今天我想吃火锅。
AI: 火...

>>> 用户: 还记得我是做什么工作的吗？
   [Milvus 命中历史]: 找到了 2 条相关记录
AI: 当然记得，你是一名建筑师！
   [已写入 Milvus]: User: 还记得我是做什么工作的吗？
...

>>> 用户: 我刚才说想吃什么？
   [Milvus 命中历史]: 找到了 2 条相关记录
AI: 你刚才说想吃火锅！
   [已写入 Milvus]: User: 我刚才说想吃什么？
AI: ...
